In [2]:
!pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 44.8 MB/s eta 0:00:0000:01


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import os
import numpy as np
import pandas as pd
import pydicom
import torch
from tqdm.auto import tqdm
from monai.transforms import (
    Compose, Spacingd, Orientationd,
    ScaleIntensityRanged, Resized, CopyItemsd, ConcatItemsd, DeleteItemsd, MapTransform,
    CropForegroundd
)
from monai.data import MetaTensor

import pandas as pd
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from monai.data import Dataset, DataLoader
from tqdm import tqdm  # 학습 진행 상황 시각화를 위해 추가
import timm
from collections import OrderedDict

# -------------------------------------------------------------------------
# 1. 전처리용 커스텀 트랜스폼
# -------------------------------------------------------------------------
class SelectiveSamplingd(MapTransform):
    def __init__(self, keys, num_slices=64):
        super().__init__(keys)
        self.num_slices = num_slices

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            img = d[key]
            s = img.shape[1]
            indices = np.linspace(0, s - 1, self.num_slices).astype(int)
            d[key] = img[:, indices, :, :]
        return d

class Preprocessor:
    def __init__(self, target_slices=64, target_size=224):
        self.transforms = Compose([

            # 1. 방향 통일 (코드 B의 장점: 해부학적 위치 고정)
            Orientationd(keys=["image"], axcodes="RAS"),
            
            # 2. 물리적 간격 표준화 (1.5mm Isotropic)
            Spacingd(keys=["image"], pixdim=(1.5, 1.5, 1.5), mode="bilinear"),
            
            # 3. 외상 특화 윈도잉 3채널 생성 (코드 A의 장점)
            CopyItemsd(keys=["image"], times=3, names=["img_soft", "img_angio", "img_bowel"]),
            
            # Soft Tissue (장기 손상용)
            ScaleIntensityRanged(keys=["img_soft"], a_min=-160, a_max=240, b_min=0.0, b_max=1.0, clip=True),
            # Angio/Blood (활성 출혈 점 강조용)
            ScaleIntensityRanged(keys=["img_angio"], a_min=-250, a_max=450, b_min=0.0, b_max=1.0, clip=True),
            # Bowel/Air (장 천공 가스 강조용)
            ScaleIntensityRanged(keys=["img_bowel"], a_min=-300, a_max=200, b_min=0.0, b_max=1.0, clip=True),
            
            # 4. 채널 결합 및 불필요 항목 삭제
            ConcatItemsd(keys=["img_soft", "img_angio", "img_bowel"], name="image"),
            DeleteItemsd(keys=["img_soft", "img_angio", "img_bowel"]),
            
            # 5. 크기 표준화 (깊이 64, 가로세로 128로 리사이징)
            # 검은 공기(배경) 잘라내기 (환자 몸통만 남김)
            # 윈도잉으로 공기를 0으로 만든 직후에 써야 합니다.
            # margin=5를 주어 피부 바깥쪽 정보가 너무 칼같이 잘리지 않게 보호합니다.
            CropForegroundd(keys=["image"], source_key="image", margin=5),
            
            # 64장 샘플링 (위의 SelectiveSamplingd 사용)
            SelectiveSamplingd(keys=["image"], num_slices=64),
            
            # 5. 가로세로만 리사이즈 (-1은 깊이(64장)를 유지하라는 뜻)
            Resized(keys=["image"], spatial_size=(-1, target_size, target_size))
        ])

    def get_valid_dicom_files(self, dcm_dir):
        """코드 A의 장점: 손상된 DICOM 파일을 사전에 걸러냄"""
        valid_slices = []

        for f in os.listdir(dcm_dir):
            path = os.path.join(dcm_dir, f)
            try:
                ds = pydicom.dcmread(path)
                _ = ds.pixel_array # RLE 디코딩 테스트
                
                if "ImagePositionPatient" in ds:
                    valid_slices.append(ds)
            except:
                continue
        
        # Z-Coordinate 기준 정렬 (해부학적 순서)
        valid_slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))

        # HU 변환 및 볼륨 생성
        vol = []
        for s in valid_slices:
            img = s.pixel_array.astype(np.float32)
            slope = valid_slices[0].RescaleSlope
            intercept = valid_slices[0].RescaleIntercept
            vol.append(img * slope + intercept)
            
        vol = np.stack(vol, axis=0)
        
        if len(valid_slices) > 1:
            z_spacing = np.abs(valid_slices[1].ImagePositionPatient[2] - valid_slices[0].ImagePositionPatient[2]) 
        else:
            z_spacing = valid_slices[0].SliceThickness
            
        curr_spacing = (
            z_spacing, 
            float(valid_slices[0].PixelSpacing[0]), 
            float(valid_slices[0].PixelSpacing[1])
        )
        
        # 2. 현재 Spacing을 기반으로 Affine 행렬(4x4) 생성
        # 1.0은 monai에서 필요한 4x4를 맞추기 위해, 원래는 3x3
        affine = np.diag(list(curr_spacing) + [1.0])

        # 3. MetaTensor 생성 (데이터 + Affine을 하나로 묶음)
        # vol: (D, H, W) -> [None] 추가하여 (C, D, H, W) 형태로 변환
        vol_meta = MetaTensor(vol[None], affine=affine)
        
        return vol_meta

# -------------------------------------------------------------------------
# 2. 실시간 데이터 로더 (Dataset)
# -------------------------------------------------------------------------
class RSNAInferenceDataset(Dataset):
    def __init__(self, df, base_dir, preprocessor):
        # series_ids 리스트를 따로 관리하지 않고, 전달받은 데이터프레임(df)을 그대로 사용합니다.
        self.df = df
        self.base_dir = base_dir
        self.preprocessor = preprocessor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # [수정 핵심] iloc를 사용하여 idx번째 행(row)을 가져옵니다.
        row = self.df.iloc[idx]
        
        # 컬럼명을 통해 안전하게 ID를 추출합니다.
        patient_id = str(int(row['patient_id']))
        series_id = str(int(row['series_id']))
        
        dcm_dir = os.path.join(self.base_dir, "test_images", patient_id, series_id)
        
        try:
            vol_meta = self.preprocessor.get_valid_dicom_files(dcm_dir)
            if vol_meta is None: 
                raise ValueError(f"Invalid DICOM at {dcm_dir}")
                
            data = self.preprocessor.transforms({"image": vol_meta})
            img = data["image"] 
            
            # (C, D, H, W) 형태의 텐서 반환
            return img.as_tensor(), int(series_id), 1 
        except Exception as e:
            # 에러 발생 시 로그 출력 (필요 시)
            # print(f"Error processing series {series_id}: {e}")
            return torch.zeros((3, 64, 224, 224)), int(series_id), 0

class Timm_Model(torch.nn.Module):
    def __init__(self, model_name='convnext_tiny', num_slices=64):
        super().__init__()
        # 특징 추출기 (ConvNeXt)
        # num_classes = 1000 (기본값): 모델의 최종 출력이 1,000개의 숫자(카테고리 점수)로 나옵니다.
        # num_classes = 0: 1,000개를 맞히는 마지막 층을 아예 없애버립니다. 대신, 그 바로 직전 단계인 **'이미지의 핵심 특징 정보(Feature Vector)'**를 그대로 출력합니다.
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        

        for param in self.backbone.parameters():
            param.requires_grad = False
            
        self.dim = self.backbone.num_features # tiny 기준 768
        self.num_slices = num_slices
        self.gated_norm = nn.LayerNorm(self.dim)

        # Position Encoding (슬라이스 번호 매기기)
        self.position_embedding = nn.Parameter(torch.zeros(1, num_slices, self.dim))
        self.position_dropout = nn.Dropout(0.1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.dim, 
            nhead=8, 
            dim_feedforward=self.dim * 2, 
            dropout=0.1, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # 어텐션 풀링: 64장 중 수상한 놈을 골라내는 '심사위원'
        self.attention_net = nn.Sequential(
            nn.Linear(self.dim, 256),
            nn.Tanh(),
            nn.Dropout(0.1), # 추가
            nn.Linear(256, 1)
        )
        
        # "종합 이상 징후 탐지" 전용 헤드 (의심 모델 역할)
        self.suspicion_head = nn.Sequential(
            nn.Linear(self.dim, 256),  # 768개를 256개의 핵심 의심 후보로 압축
            nn.LayerNorm(256),  # 학습을 안정적으로 만들어줌
            nn.ReLU(),            # 중요한 의심 신호만 통과시킴
            nn.Dropout(0.2),      # 과적합 방지 (너무 예민해지는 것 방지)
            nn.Linear(256, 2)     # 최종 경보 [정상, 이상]
        )

        # "정밀 병명 분류" 전용 헤드 (분류 모델 역할)
        # 장기별 결과 2 or 3개 도출
        self.organ_heads = nn.ModuleDict({
            'bowel': nn.Linear(self.dim, 2),
            'extravasation': nn.Linear(self.dim, 2),
            'liver': nn.Linear(self.dim, 3),
            'kidney': nn.Linear(self.dim, 3),
            'spleen': nn.Linear(self.dim, 3)
        })

    # [입력 데이터] (Batch, 64 Slices, 3, 128, 128)
    #     ↓
    # ==========================================================
    # 1. [Backbone: ConvNeXt-Tiny] -> "시각 신경 (이미지 스캐너)"
    # - 64장의 슬라이스를 각각 스캔하여 768차원의 특징 추출
    # - (64, 3, 128, 128) -> (64, 768)
    # ==========================================================
    #     ↓
    # 2. [Position Embedding] -> "인덱스 부여 (공간 좌표)"
    # - 각 특징에 "이건 1번(머리), 이건 64번(골반)"이라는 위치 정보 주입
    # ==========================================================
    #     ↓
    # 3. [Transformer Encoder] -> "종합 분석 (슬라이스 간 대화)"
    # - 64장의 특징들이 서로 정보를 교환하며 전후 맥락 파악
    # - "5번 슬라이스의 상처가 10번까지 이어지네? 큰 부상이다!"
    # ==========================================================
    #     ↓
    # 4. [Attention Pooling] -> "심사위원 (결정적 증거 포착)"
    # - 64장 중 가장 수상한(부상이 의심되는) 슬라이스에 높은 점수 부여
    # - 64개의 특징을 단 1개의 '필살기 특징 벡터'로 압축 (1, 768)
    # ==========================================================
    #     ↓
    # ==========================================================
    # 5. [suspicion_head] -> "응급의학과 의사" (전체 부상 유무 판단)
    # - [부상 확률 (injury_prob)] (0.0 ~ 1.0)
    # ==========================================================
    #     ↓
    # ==========================================================
    # [organ_heads] -> "전문의 진단 (최종 판단)"
    # - Bowel, Liver, Kidney, Spleen 등 정밀 진단
    # - 부상 확률을 곱하기 때문에 부상에 결과에 영향을 받음
    # ==========================================================
    def forward(self, x):
        # 2.5D 방식으로 전체 슬라이스 훑기
        # x shape: (Batch, 64, 3, 128, 128)
        b, s, c, h, w = x.shape
        
        chunk_size = 8 # 한 번에 처리할 슬라이스 개수 (메모리에 따라 조절)
        all_features = []
        
        for i in range(0, s, chunk_size):
            # x_chunk: (Batch, 16, 3, 128, 128)
            x_chunk = x[:, i : i + chunk_size] 
            
            # 2D 연산을 위해 일시적으로 배치 차원으로 합침
            x_chunk = x_chunk.reshape(-1, c, h, w) # (Batch*16, 3, 128, 128)
            
            # 백본 통과 (이 순간 메모리 사용량이 chunk_size만큼으로 제한됨)
            feat_chunk = self.backbone(x_chunk) # (Batch*16, 768)
            
            # 다시 슬라이스 차원 분리 후 리스트에 저장
            feat_chunk = feat_chunk.view(b, -1, self.dim) 

            all_features.append(feat_chunk)

        # 모든 특징 합치기
        features = torch.cat(all_features, dim=1) # (Batch, 64, 768)
    
        # --- [Step 2] Position Encoding: 위치 정보 주입 ---
        # 데이터가 정렬되어 들어와도, 모델이 이를 '좌표'로 인식하게 함
        features = features + self.position_embedding
        features = self.position_dropout(features)

        # --- [Step 3] Transformer Encoder: 슬라이스 간 상호작용 ---
        # 64장의 슬라이스가 서로의 정보를 참조하여 입체적인 특징으로 진화
        features = self.transformer_encoder(features) # (Batch, 64, 768)

        # Attention Pooling으로 '이상 지점' 증폭
        # 각 슬라이스의 수상함 점수 계산
        att_scores = self.attention_net(features) # (B, 64, 1)
        
         # 점수를 0~1 사이 비중(가중치)으로 변환
        att_weights = F.softmax(att_scores, dim=1) # (B, S, 1)
        
        # 가중치를 곱해서 하나로 합침 (가장 수상한 슬라이스 정보가 증폭됨)
        combined = torch.sum(features * att_weights, dim=1) # (B, 768)

        # 결과 도출
        # 1. 먼저 "부상 유무"를 판단합니다.
        injury_logits = self.suspicion_head(combined) # (B, 2)
        # 부상일 확률(Probability)을 구합니다.
        injury_prob = torch.softmax(injury_logits, dim=1)[:, 1:2] # (B, 1)

        # 2. [핵심] 부상 확률을 장기별 특징에 곱해줍니다 (Gating)
        # 부상이 아닐 것 같으면(0에 가까우면) 장기별 점수들도 0에 가까워지도록 강제합니다.
        gated_features = self.gated_norm(combined * injury_prob)

        # 3. 정밀 진단은 이 게이트를 통과한 특징으로 수행합니다.
        out = {k: head(gated_features) for k, head in self.organ_heads.items()}
        out['any_injury'] = injury_logits

        return out

# -------------------------------------------------------------------------
# 2. 모델 로드 (본인의 모델 클래스로 교체 필요)
# -------------------------------------------------------------------------
BASE_DIR = '/kaggle/input/rsna-2023-abdominal-trauma-detection/'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MONAI_MODEL_PATH = '/kaggle/input/ct-v6/pytorch/default/5/monai_ct_convnext_v6_2_ep17.pth'
SAVE_DIR = '/kaggle/working/'

model = Timm_Model(model_name='convnext_tiny').to(DEVICE)
if os.path.exists(MONAI_MODEL_PATH):
    print(f"==> 로드 중: {MONAI_MODEL_PATH}")
    checkpoint = torch.load(MONAI_MODEL_PATH, map_location=DEVICE)
        
    # [수정] module. 접두어 제거 후 로드
    state_dict = checkpoint['model']
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        new_state_dict[name] = v
    model.load_state_dict(new_state_dict)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# -------------------------------------------------------------------------
# 3. 추론 진행
# -------------------------------------------------------------------------
test_meta = pd.read_csv(f'{BASE_DIR}test_series_meta.csv')

preprocessor = Preprocessor(target_size=224)

# Dataset 생성 (target_size=224 확인)
test_dataset = RSNAInferenceDataset(test_meta, BASE_DIR, preprocessor)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2) # 메모리 고려하여 배치 조정

predictions = []
with torch.no_grad():
    for imgs, s_ids, flags in tqdm(test_loader, desc="추론 중"):
        # imgs shape: [B, 3, 64, 224, 224]
        if not flags.any(): # 모든 배치가 전처리 실패한 경우
            continue
            
        inputs = imgs.permute(0, 2, 1, 3, 4).to(DEVICE) # [B, 64, 3, 224, 224]
        outputs = model(inputs)
        
        # 배치를 돌며 결과 저장 (batch_size=1이라도 루프를 도는 것이 안전함)
        for i in range(len(s_ids)):
            if flags[i] == 0: continue
            
            res = {'series_id': int(s_ids[i])}
            for organ, logits in outputs.items():
                if organ == 'any_injury': continue
                
                # logits[i]를 사용하여 해당 배치의 확률 계산
                p = torch.softmax(logits[i], dim=0).cpu().numpy()
                
                if p.shape[0] == 2:
                    res[f'{organ}_healthy'], res[f'{organ}_injury'] = p[0], p[1]
                else:
                    res[f'{organ}_healthy'], res[f'{organ}_low'], res[f'{organ}_high'] = p[0], p[1], p[2]
            predictions.append(res)

# 4) 최종 제출 파일 (Post-processing)
pred_df = pd.DataFrame(predictions)
final_df = test_meta[['patient_id', 'series_id']].merge(pred_df, on='series_id', how='left')

fill_values = {
    'bowel_healthy': 1.0, 'bowel_injury': 0.0, 'extravasation_healthy': 1.0, 'extravasation_injury': 0.0,
    'kidney_healthy': 1.0, 'kidney_low': 0.0, 'kidney_high': 0.0,
    'liver_healthy': 1.0, 'liver_low': 0.0, 'liver_high': 0.0,
    'spleen_healthy': 1.0, 'spleen_low': 0.0, 'spleen_high': 0.0
}
final_df = final_df.fillna(value=fill_values)
sub_df = final_df.groupby('patient_id').mean().reset_index()

final_columns = [
    'patient_id', 'bowel_healthy', 'bowel_injury', 'extravasation_healthy', 
    'extravasation_injury', 'kidney_healthy', 'kidney_low', 'kidney_high', 
    'liver_healthy', 'liver_low', 'liver_high', 'spleen_healthy', 'spleen_low', 'spleen_high'
]
sub_df[final_columns].to_csv('/kaggle/working/submission.csv', index=False)
print("submission.csv 완료!")

==> 로드 중: /kaggle/input/ct-v6/pytorch/default/5/monai_ct_convnext_v6_2_ep17.pth


추론 중: 100%|██████████| 3/3 [00:35<00:00, 11.70s/it]

--- Submission.csv 생성 완료 ---
   patient_id  bowel_healthy  bowel_injury  extravasation_healthy  \
0       48843        0.79185       0.20815               0.755512   
1       50046        0.79185       0.20815               0.755512   
2       63706        0.79185       0.20815               0.755512   

   extravasation_injury  kidney_healthy  kidney_low  kidney_high  \
0              0.244488        0.880041     0.04111     0.078849   
1              0.244488        0.880041     0.04111     0.078849   
2              0.244488        0.880041     0.04111     0.078849   

   liver_healthy  liver_low  liver_high  spleen_healthy  spleen_low  \
0       0.875507   0.046566    0.077927        0.869498    0.055387   
1       0.875507   0.046566    0.077927        0.869498    0.055387   
2       0.875507   0.046566    0.077927        0.869498    0.055387   

   spleen_high  
0     0.075115  
1     0.075115  
2     0.075115  
